# Lesson 11: Differentiation and Edge Detection

An edge is, informally, a place where intensity changes quickly. That's a statement about a *derivative*. This lesson builds up discrete image derivatives &mdash; Prewitt, Sobel, and Scharr operators &mdash; as convolution kernels (Lesson 9), then uses them inside the classic **Canny** edge detector.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## From derivatives to gradients

Treat the image as a function $I(x,y)$. Its gradient $\nabla I = (I_x, I_y)$ points in the direction of steepest intensity increase. Two numbers summarize it at each pixel:

- **magnitude** $\|\nabla I\| = \sqrt{I_x^2 + I_y^2}$ &mdash; how sharp the change is (large at edges, near zero on flat regions)
- **direction** $\theta = \mathrm{atan2}(I_y, I_x)$ &mdash; which way intensity is increasing fastest (perpendicular to the edge)

$I_x$ and $I_y$ are themselves computed by convolving the image with small derivative kernels.

In [ ]:
def make_test_image(size=200):
    img = np.zeros((size, size), dtype=np.uint8)
    cv2.rectangle(img, (30, 30), (100, 100), 200, -1)
    cv2.circle(img, (140, 140), 40, 150, -1)
    pts = np.array([[150, 30], [190, 90], [110, 90]], dtype=np.int32)
    cv2.fillPoly(img, [pts], 25)  # low-contrast triangle, for the threshold demo later
    return img

img = make_test_image()
plt.imshow(img, cmap='gray')
plt.title('Test image (edges at several orientations)')
plt.axis('off')
plt.show()

## The naive derivative: just a difference

The simplest $I_x$ estimate is a 1D central-difference kernel $[-1,\ 0,\ 1]$. It works, but with no smoothing in the perpendicular direction, it's maximally sensitive to noise.

In [ ]:
simple_x = np.array([[-1, 0, 1]], dtype=np.float64)
simple_y = simple_x.T

gx = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_x)
gy = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_y)
magnitude = np.sqrt(gx**2 + gy**2)

plt.imshow(magnitude, cmap='gray')
plt.title('Gradient magnitude, simple [-1, 0, 1] kernel')
plt.axis('off')
plt.show()

## Prewitt, Sobel, and Scharr: derivative + smoothing

Real operators combine a difference in one direction with a *smoothing* average in the perpendicular direction &mdash; this is what makes them robust to noise. They differ only in how they weight that smoothing:

| Operator | $x$-kernel | Perpendicular weighting |
|---|---|---|
| **Prewitt** | $\begin{bmatrix}-1&0&1\\-1&0&1\\-1&0&1\end{bmatrix}$ | uniform: 1, 1, 1 |
| **Sobel** | $\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}$ | binomial: 1, 2, 1 |
| **Scharr** | $\begin{bmatrix}-3&0&3\\-10&0&10\\-3&0&3\end{bmatrix}$ | 3, 10, 3 |

Each $y$-kernel is just the transpose of its $x$-kernel. Sobel's binomial weights approximate a small Gaussian, which is why it's the most commonly used default. Scharr's weights were chosen (numerically optimized) specifically to make the *estimated gradient direction* as rotationally accurate as possible &mdash; i.e. as close to a true continuous derivative as an integer $3\times3$ kernel can get &mdash; which matters most in applications like optical flow that depend on precise gradient angles rather than just edge location.

In [ ]:
prewitt_x = np.array([[-1, 0, 1]] * 3, dtype=np.float64)
prewitt_y = prewitt_x.T

operators = {
    'Simple': (simple_x, simple_y),
    'Prewitt': (prewitt_x, prewitt_y),
    'Sobel': None,   # use cv2.Sobel directly below
    'Scharr': None,  # use cv2.Scharr directly below
}

img_f = img.astype(np.float64)
magnitudes = {}
magnitudes['Simple'] = magnitude
magnitudes['Prewitt'] = np.sqrt(cv2.filter2D(img_f, cv2.CV_64F, prewitt_x)**2 +
                                 cv2.filter2D(img_f, cv2.CV_64F, prewitt_y)**2)
magnitudes['Sobel'] = np.sqrt(cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)**2 +
                               cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)**2)
magnitudes['Scharr'] = np.sqrt(cv2.Scharr(img_f, cv2.CV_64F, 1, 0)**2 +
                                cv2.Scharr(img_f, cv2.CV_64F, 0, 1)**2)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, (name, mag) in zip(axes, magnitudes.items()):
    ax.imshow(mag, cmap='gray')
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

On a clean, low-noise image like this, the four results look nearly identical &mdash; the practical differences between them are subtle and show up mainly under noise or when precise sub-pixel angle matters, not as a visibly different edge map.

### Noise robustness: why the perpendicular smoothing matters

To make the benefit of that perpendicular smoothing concrete, we measure each kernel's response to pure noise on a flat (edge-free) region. To compare fairly, we first rescale each kernel so it has the *same gain* on an ideal step edge (i.e. divide by the sum of its positive weights) &mdash; otherwise a kernel with larger raw coefficients would look noisier just from its overall scale, not its shape.

In [ ]:
rng = np.random.default_rng(0)
flat_noisy = np.full((150, 150), 128.0) + rng.normal(0, 15, (150, 150))

kernels = {
    'Simple [-1,0,1]': simple_x,
    'Prewitt': prewitt_x,
    'Sobel': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64),
    'Scharr': np.array([[-3, 0, 3], [-10, 0, 10], [-3, 0, 3]], dtype=np.float64),
}

print(f'{"kernel":>18} {"noise std (gain-normalized)":>30}')
for name, k in kernels.items():
    gain = k[k > 0].sum()
    response = cv2.filter2D(flat_noisy, cv2.CV_64F, k / gain)
    print(f'{name:>18} {response.std():>30.2f}')

The plain difference kernel is noticeably noisier than any of the $3\times3$ operators &mdash; averaging over 3 rows (or columns) while differencing cuts down the noise response substantially, which is exactly the point of building smoothing into the derivative kernel.

## From gradients to edges: the Canny detector

Simply thresholding gradient magnitude gives thick, noisy edge blobs. The **Canny** edge detector refines this with a multi-stage pipeline:

1. **Smooth** with a Gaussian, to suppress noise before differentiating.
2. **Compute gradients** (Sobel, internally) to get magnitude and direction at every pixel.
3. **Non-maximum suppression**: at each pixel, keep the gradient magnitude only if it's a local maximum *along the gradient direction* &mdash; this thins wide gradient ridges down to single-pixel-wide lines.
4. **Double thresholding + hysteresis**: pixels above a high threshold are definite edges; pixels below a low threshold are discarded; pixels in between are kept only if they connect to a definite edge. This links up weak-but-real edge segments while suppressing isolated noise responses.

In [ ]:
noisy_img = np.clip(img_f + rng.normal(0, 8, img.shape), 0, 255).astype(np.uint8)

naive_edges = (magnitudes['Sobel'] > 150).astype(np.uint8) * 255  # crude: just threshold |gradient|
canny_edges = cv2.Canny(img, threshold1=80, threshold2=160)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, im, title in zip(axes, [img, naive_edges, canny_edges],
                          ['Original', 'Naive: threshold |gradient|\n(thick, blobby)', 'Canny\n(thin, connected)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Threshold sensitivity

Canny's two thresholds trade off completeness against noise. Too low, and noise gets picked up as spurious edges; too high, and real (but low-contrast) edges are missed.

In [ ]:
threshold_pairs = [(20, 60), (80, 160), (150, 220)]

fig, axes = plt.subplots(1, len(threshold_pairs), figsize=(10, 3.5))
for ax, (lo, hi) in zip(axes, threshold_pairs):
    edges = cv2.Canny(noisy_img, lo, hi)
    ax.imshow(edges, cmap='gray')
    ax.set_title(f'thresholds=({lo}, {hi})', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

With low thresholds on the noisy image, speckle noise survives as spurious edge fragments scattered everywhere. At the highest thresholds, the low-contrast triangle's boundary disappears completely, while the higher-contrast rectangle and circle edges survive — a real example of Canny's threshold trading off noise rejection against sensitivity to faint-but-genuine edges.

### Exercise

1. Increase the noise standard deviation in `flat_noisy` and re-run the noise-robustness comparison. Does the relative ordering of the four kernels change?
2. `cv2.Canny` accepts an `apertureSize` parameter for its internal Sobel step (default 3). Try `apertureSize=5` and compare the result to the default on the noisy image.
3. Canny's hysteresis step needs a *connected* path of above-low-threshold pixels between a weak edge and a strong one. Construct a small binary example (by hand, as a NumPy array) where a real edge is broken into two segments with a 1-pixel gap, and confirm that hysteresis fails to link them even though a human viewer would clearly see one edge.